Oracle AI Data Platform v1.0

Copyright © 2026, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# Notion Ingestion (Full Refresh and CDC)

Lands Notion **pages**, **data sources**, **blocks** and **users** in Delta tables in the catalog and schema named in `notion_ingest.yaml`.

- **CDC (default):** reads only what changed since the last successful run, using a per-object watermark kept in `notion_sync_state`.
- **Full:** reloads everything and removes what no longer exists in Notion.

**Before you run**

1. Create a Notion internal integration, copy its secret, and share the pages and databases you want with it.
2. Store the secret in the AIDP Credential Store and set `notion.credential_name` in the config.
3. Install `requirements.txt` as a cluster library.
4. Copy `notion_ingest.sample.yaml` to `notion_ingest.yaml` next to this notebook and edit the target catalog and schema.

See `README.md` in this folder for details and known limits.

## 1. Locate the sample folder

In [ ]:
import os
import sys

# Folder that contains this notebook, notion_connector/ and notion_ingest.yaml.
SAMPLE_DIR = "/Workspace/<path-to>/notion"

if SAMPLE_DIR not in sys.path:
    sys.path.insert(0, SAMPLE_DIR)

from notion_connector import format_summary, load_config, raise_on_failure, run

## 2. Load the configuration

In [ ]:
config = load_config(os.path.join(SAMPLE_DIR, "notion_ingest.yaml"))

print("target :", config.target.qualified_schema)
print("objects:", ", ".join(config.sync.objects))
print("mode   :", config.sync.mode)

## 3. Optional per-run override

When this notebook runs as a job, a job parameter `MODE` (`cdc` or `full`) overrides `sync.mode` for that run only. Interactive runs use the config.

In [ ]:
def job_parameter(name, default=None):
    try:
        # oidlUtils is provided by the AIDP runtime - do not import it.
        return oidlUtils.parameters.getParameter(name, default)  # noqa: F821
    except NameError:
        return default

MODE = job_parameter("MODE")
print("mode override:", MODE or "(none)")

## 4. Read the Notion token

In [ ]:
def notion_token(settings):
    if settings.credential_name:
        try:
            # aidputils is provided by the AIDP runtime - do not import it.
            return aidputils.secrets.get(name=settings.credential_name, key=settings.credential_key)  # noqa: F821
        except NameError:
            pass  # not on AIDP: fall back to the environment variable
    token = os.environ.get(settings.token_env)
    if not token:
        raise RuntimeError(
            f"No Notion token found. Set notion.credential_name in the config, "
            f"or export {settings.token_env}."
        )
    return token

token = notion_token(config.notion)
print("token read (not shown)")

## 5. Run the sync

In [ ]:
summaries = run(spark, config, token, mode_override=MODE)  # noqa: F821 - spark is provided by the notebook

print(format_summary(summaries))
raise_on_failure(summaries)

## 6. Look at the result

```sql
SELECT id, title, last_edited_time, in_trash FROM <catalog>.<schema>.pages ORDER BY last_edited_time DESC LIMIT 20;

-- Text of a page, in reading order
SELECT depth, position, type, raw_json FROM <catalog>.<schema>.blocks WHERE page_id = '<page id>';

-- Watermarks and the outcome of the last run per object
SELECT * FROM <catalog>.<schema>.notion_sync_state;
```